# Files, Paths, JSON, CSV, Dates and Time Zones

Notebook for **Chapter 5** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Files and Paths withpathlib

In [ ]:
from pathlib import Path
import tempfile, shutil, glob, os

with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "logs").mkdir()
    (root / "logs" / "app.log").write_text("line1\nline2\nline3\n", encoding="utf-8")
    (root / "logs" / "old.log").write_text("x\n", encoding="utf-8")
    (root / "notes.txt").write_text("hello", encoding="utf-8")
    p = root / "logs" / "app.log"
    print(p.name, p.stem, p.suffix, p.parent.name)
    print(p.exists(), p.is_file(), p.stat().st_size)
    print(sorted(x.name for x in root.rglob("*.log")))
    print(sorted(os.path.basename(x) for x in glob.glob(str(root / "logs" / "*.log"))))
    with open(p, encoding="utf-8") as f:
        for n, line in enumerate(f, start=1):
            print(n, line.rstrip())

app.log app .log logs
True True 18
['app.log', 'old.log']
['app.log', 'old.log']
1 line1
2 line2
3 line3


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
open("/no/such/dir/data.csv", encoding="utf-8")

FileNotFoundError: [Errno 2] No such file or directory: '/no/such/dir/data.csv'


In [ ]:
from pathlib import Path
import tempfile, shutil, os

with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "logs").mkdir()
    (root / "logs" / "app.log").write_text("line1\n", encoding="utf-8")
    (root / "notes.txt").write_text("hello", encoding="utf-8")
    shutil.copy(root / "logs" / "app.log", root / "backup.log")
    print(sorted(x.name for x in root.iterdir()))
    shutil.rmtree(root / "logs")
    print(sorted(x.name for x in root.iterdir()))
    with open(root / "a.txt", "a", encoding="utf-8") as f:
        f.write("one\n")
    with open(root / "a.txt", "a", encoding="utf-8") as f:
        f.write("two\n")
    print(repr((root / "a.txt").read_text()))
    (root / "b.bin").write_bytes(bytes([72, 105, 255]))
    print((root / "b.bin").read_bytes())
print(Path("data") / "raw" / "x.csv")
print(os.sep, os.path.join("data", "raw", "x.csv"))

['backup.log', 'logs', 'notes.txt']
['backup.log', 'notes.txt']
'one\ntwo\n'
b'Hi\xff'
data/raw/x.csv
/ data/raw/x.csv


## 5.2 JSON

In [ ]:
import json
d = {"name": "Asha", "age": 29, "skills": ["python", "sql"],
     "active": True, "manager": None, "score": 91.5}
s = json.dumps(d)
print(s)
print(json.dumps(d, indent=2, sort_keys=True))
back = json.loads(s)
print(back == d, type(back["active"]), back["manager"])
print(json.loads('{"1": "a"}'), json.dumps({1: "a"}))
print(json.dumps((1, 2)), json.loads(json.dumps((1, 2))))

{"name": "Asha", "age": 29, "skills": ["python", "sql"], "active": true, "manager": null, "score": 91.5}
{
  "active": true,
  "age": 29,
  "manager": null,
  "name": "Asha",
  "score": 91.5,
  "skills": [
    "python",
    "sql"
  ]
}
True <class 'bool'> None
{'1': 'a'} {"1": "a"}
[1, 2] [1, 2]


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import json
from datetime import datetime
try:
    json.dumps({"when": datetime(2026, 1, 5)})
except TypeError as e:
    print("TypeError:", e)
print(json.dumps({"when": datetime(2026, 1, 5)}, default=lambda o: o.isoformat()))
try:
    json.loads("{'a': 1}")
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)
print(json.dumps({"city": "Chennai", "word": "café"}))
print(json.dumps({"word": "café"}, ensure_ascii=False))

TypeError: Object of type datetime is not JSON serializable
{"when": "2026-01-05T00:00:00"}
JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
{"city": "Chennai", "word": "caf\u00e9"}
{"word": "café"}


In [ ]:
import json
lines = '{"id":1,"delivery_minutes":20}\n{"id":2,"delivery_minutes":55}\n{"id":3,"delivery_minutes":15}\n'
rows = [json.loads(l) for l in lines.splitlines()]
print(sum(r["delivery_minutes"] for r in rows) / len(rows))
nested = {"user": {"address": {"city": "Pune"}}}
print(nested.get("user", {}).get("address", {}).get("zip", "n/a"))

30.0
n/a


## 5.3 CSV

In [ ]:
import csv, io
from collections import defaultdict
text = """order_id,city,amount
1,Pune,250.50
2,Chennai,1200
3,"Delhi, North",75.25
"""
rows = list(csv.DictReader(io.StringIO(text)))
print(rows[2])
print(sum(float(x["amount"]) for x in rows))
for row in csv.reader(io.StringIO(text)):
    print(row)
agg = defaultdict(float)
for x in rows:
    agg[x["city"]] += float(x["amount"])
print(dict(agg))

{'order_id': '3', 'city': 'Delhi, North', 'amount': '75.25'}
1525.75
['order_id', 'city', 'amount']
['1', 'Pune', '250.50']
['2', 'Chennai', '1200']
['3', 'Delhi, North', '75.25']
{'Pune': 250.5, 'Chennai': 1200.0, 'Delhi, North': 75.25}


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
total = 0
for r in rows:
    total += r["amount"]

TypeError: unsupported operand type(s) for +=: 'int' and 'str'


In [ ]:
import csv, io
out = io.StringIO()
w = csv.DictWriter(out, fieldnames=["order_id", "city", "amount"], lineterminator="\n")
w.writeheader()
w.writerow({"order_id": 4, "city": 'Goa "Beach"', "amount": 10})
w.writerow({"order_id": 5, "city": "Kochi, Kerala", "amount": 20})
print(out.getvalue())
print(next(csv.reader(io.StringIO("a;b;c"), delimiter=";")))
print(sorted(["10", "9", "100"]), sorted(["10", "9", "100"], key=int))

order_id,city,amount
4,"Goa ""Beach""",10
5,"Kochi, Kerala",20

['a', 'b', 'c']
['10', '100', '9'] ['9', '10', '100']


## 5.4 Dates, Times and Time Zones

In [ ]:
from datetime import datetime, date, timedelta
d = date(2026, 3, 15)
print(d, d.weekday(), d.strftime("%A, %d %B %Y"))
dt = datetime(2026, 3, 15, 9, 30)
print(dt.isoformat(), dt + timedelta(days=20, hours=5))
print(date(2026, 12, 25) - date(2026, 11, 7))
p = datetime.strptime("07/11/2026 14:05", "%d/%m/%Y %H:%M")
print(p, p.year)
print(datetime.fromisoformat("2026-11-07T14:05:00+05:30"))
print(date(2026, 2, 28) + timedelta(days=1))
start = datetime(2026, 1, 1, 10, 0, 0)
end = datetime(2026, 1, 1, 10, 0, 2, 350000)
print((end - start).total_seconds(), end - start)

2026-03-15 6 Sunday, 15 March 2026
2026-03-15T09:30:00 2026-04-04 14:30:00
48 days, 0:00:00
2026-11-07 14:05:00 2026
2026-11-07 14:05:00+05:30
2026-03-01
2.35 0:00:02.350000


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from datetime import datetime
datetime.strptime("07/11/2026", "%m/%d/%Y")   # reads 11 July
datetime.strptime("25/11/2026", "%m/%d/%Y")

ValueError: time data '25/11/2026' does not match format '%m/%d/%Y'


In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
ist = ZoneInfo("Asia/Kolkata")
ny = ZoneInfo("America/New_York")
t = datetime(2026, 7, 1, 9, 0, tzinfo=ist)
print(t, t.astimezone(ny), t.astimezone(timezone.utc))

2026-07-01 09:00:00+05:30 2026-06-30 23:30:00-04:00 2026-07-01 03:30:00+00:00


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
ny = ZoneInfo("America/New_York")
a = datetime(2026, 3, 8, 1, 30, tzinfo=ny)
b = a + timedelta(hours=1)
print(a, b, b.utcoffset())
print(b.astimezone(timezone.utc) - a.astimezone(timezone.utc))
try:
    datetime(2026, 1, 1) - datetime(2026, 1, 1, tzinfo=timezone.utc)
except TypeError as e:
    print("TypeError:", e)
print(datetime.fromtimestamp(1_800_000_000, tz=timezone.utc))
print(datetime(2026, 1, 1, tzinfo=timezone.utc).timestamp())

2026-03-08 01:30:00-05:00 2026-03-08 02:30:00-05:00 -1 day, 19:00:00
1:00:00
TypeError: can't subtract offset-naive and offset-aware datetimes
2027-01-15 08:00:00+00:00
1767225600.0


## 5.5 Configuration: Environment Variables,.envFiles, YAML and TOML

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import os, tempfile
from pathlib import Path

def load_dotenv(path):
    env = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

with tempfile.TemporaryDirectory() as d:
    f = Path(d) / ".env"
    f.write_text('# demo\nAPP_ENV=dev\nDB_HOST="localhost"\nDB_PORT=5432\nAPI_KEY=fake-key-123\n')
    cfg = load_dotenv(f)
print(cfg)
port = int(os.environ.get("DB_PORT", cfg["DB_PORT"]))
print(port + 1)
os.environ["APP_ENV"] = "prod"
merged = {**cfg, **{k: v for k, v in os.environ.items() if k in cfg}}
print(merged["APP_ENV"], merged["DB_HOST"])
try:
    os.environ["REQUIRED_X"]
except KeyError as e:
    print("KeyError", e)

{'APP_ENV': 'dev', 'DB_HOST': 'localhost', 'DB_PORT': '5432', 'API_KEY': 'fake-key-123'}
5433
prod localhost
KeyError 'REQUIRED_X'


In [ ]:
import tomllib
print(tomllib.loads('[db]\nhost="h"\nport=5432\n'))

{'db': {'host': 'h', 'port': 5432}}


In [ ]:
import os
print(bool(os.environ.get("DEBUG_X", "False")))

True


## 5.6 Handling Archives, Temporary Files, Deprecated Time Calls and Pickle

Real data work moves compressed files around, writes scratch files, and keeps old code alive. This section covers the standard-library tools for that and two traps: `datetime.utcnow()` and `pickle`.

In [ ]:
import gzip, shutil, tempfile, zipfile, warnings
from datetime import datetime, timezone
from pathlib import Path
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    datetime.utcnow()
print(w[0].category.__name__)
print(datetime.now(timezone.utc).tzinfo)
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "data.csv").write_text("id,amount\n1,10\n2,20\n" * 200)
    raw = (root / "data.csv").stat().st_size
    with open(root / "data.csv", "rb") as src, gzip.open(root / "data.csv.gz", "wb") as dst:
        shutil.copyfileobj(src, dst)
    gz = (root / "data.csv.gz").stat().st_size
    print(raw, gz, gz < raw)
    with gzip.open(root / "data.csv.gz", "rt") as f:
        print(f.readline().strip(), f.readline().strip())
    with zipfile.ZipFile(root / "bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(root / "data.csv", arcname="data.csv")
        z.writestr("notes/readme.txt", "hello")
    with zipfile.ZipFile(root / "bundle.zip") as z:
        print(z.namelist())
        print(z.read("notes/readme.txt").decode())
    copy = shutil.copy(root / "data.csv", root / "backup.csv")
    print(Path(copy).name, sorted(p.name for p in root.iterdir()))
print(root.exists())

DeprecationWarning
UTC
4000 77 True
id,amount 1,10
['data.csv', 'notes/readme.txt']
hello
backup.csv ['backup.csv', 'bundle.zip', 'data.csv', 'data.csv.gz']
False


In [ ]:
import pickle, json
data = {"user": "ada", "scores": [1, 2, 3]}
print(pickle.loads(pickle.dumps(data)) == data)
class Evil:
    def __reduce__(self):
        return (print, ("code ran during unpickling",))
pickle.loads(pickle.dumps(Evil()))
print(json.loads(json.dumps(data)) == data)

True
code ran during unpickling
True


## Chapter recap

## Practice set

Problem 5.P.1 What is `Path("/data/raw/sales_2026.csv").suffix`?

In [ ]:
def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

In [ ]:
import json, tempfile
from pathlib import Path
def load_config(path, defaults):
    cfg = dict(defaults)
    p = Path(path)
    if p.exists():
        cfg.update(json.loads(p.read_text()))
    return cfg
with tempfile.TemporaryDirectory() as d:
    f = Path(d) / "c.json"
    print(load_config(f, {"port": 80, "debug": False}))
    f.write_text('{"port": 8080}')
    print(load_config(f, {"port": 80, "debug": False}))

{'port': 80, 'debug': False}
{'port': 8080, 'debug': False}


In [ ]:
from datetime import datetime, timezone, timedelta
t = datetime(2026, 3, 8, 23, 30, tzinfo=timezone.utc)
print((t + timedelta(hours=1)).isoformat())
print(datetime.fromisoformat("2026-03-08T23:30:00+00:00") == t)

2026-03-09T00:30:00+00:00
True


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch05.html